In [1]:
# Célula 1 (revisada) — DuckDB + MinIO com Secrets Manager
import duckdb

con = duckdb.connect("../data/curso_analytics.duckdb")
con.execute("INSTALL httpfs; LOAD httpfs;")

# Segredo TEMPORÁRIO: vale para esta conexão quando feita entre containers Docker. Rode esta célula sempre que reabrir o notebook.
con.execute("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin123',
        ENDPOINT 'host.docker.internal:9000',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    );
""")

# Segredo Temporário usando localhost: vale para as conexões feitas pelo host. Rode esta célula apenas uma vez.
#con.execute("""
#    CREATE OR REPLACE SECRET minio (
#        TYPE s3,
#        KEY_ID 'minioadmin',
#        SECRET 'minioadmin123',
#        ENDPOINT 'localhost:9000',
#        URL_STYLE 'path',
#        USE_SSL false,
#        REGION 'us-east-1'
#    );
#""")

print(con.sql("SELECT name, type, persistent FROM duckdb_secrets()"))

┌─────────┬─────────┬────────────┐
│  name   │  type   │ persistent │
│ varchar │ varchar │  boolean   │
├─────────┼─────────┼────────────┤
│ minio   │ s3      │ false      │
└─────────┴─────────┴────────────┘



In [2]:
# Célula 2 (revisada) — Catálogo Iceberg (SQLite local + MinIO)
import io, os, time
from datetime import datetime, timezone

import boto3
import pandas as pd
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

os.makedirs("../data/iceberg_catalog", exist_ok=True)

MINIO = {
    "s3.endpoint": "http://host.docker.internal:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin123",
    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
}

#MINIO = {
#    "s3.endpoint": "http://localhost:9000",
#    "s3.access-key-id": "minioadmin",
#    "s3.secret-access-key": "minioadmin123",
#    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
#}


catalog = SqlCatalog(
    "curso_lakehouse",
    **{"uri": "sqlite:///../data/iceberg_catalog/catalog.db", "warehouse": "s3://silver", **MINIO},
)
catalog.create_namespace_if_not_exists("bronze_processado")
print("Namespaces:", catalog.list_namespaces())


Namespaces: [('bronze_processado',)]


In [3]:
# Célula 3 — Schema fortemente tipado + particionamento oculto (hidden partitioning)
from pyiceberg.schema import Schema
from pyiceberg.types import (NestedField, LongType, IntegerType, StringType,
                             DoubleType, TimestampType, BooleanType)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import MonthTransform

schema_pedidos = Schema(
    NestedField(1,  "id_pedido",         LongType(),      required=True),
    NestedField(2,  "id_cliente",        LongType(),      required=True),
    NestedField(3,  "id_produto",        LongType(),      required=True),
    NestedField(4,  "categoria",         StringType()),
    NestedField(5,  "valor_unitario",    DoubleType()),
    NestedField(6,  "quantidade",        IntegerType()),
    NestedField(7,  "valor_total",       DoubleType()),
    NestedField(8,  "status_pedido",     StringType()),
    NestedField(9,  "regiao",            StringType()),
    NestedField(10, "data_pedido",       TimestampType()),
    NestedField(11, "avaliacao_cliente", IntegerType()),
    NestedField(12, "_fonte",            StringType()),
    NestedField(13, "_ingerido_em",      StringType()),
    identifier_field_ids=[1],            # chave lógica: usada pelo upsert
)

# O particionamento é declarado sobre uma TRANSFORMAÇÃO da coluna (mês de data_pedido).
# Quem consulta filtra por data_pedido e não precisa conhecer a estrutura de pastas.
spec_particao = PartitionSpec(
    PartitionField(source_id=10, field_id=1000, transform=MonthTransform(), name="data_pedido_mes")
)

NOME_TABELA = "bronze_processado.pedidos_ecommerce"
tabela = catalog.create_table_if_not_exists(
    identifier=NOME_TABELA,
    schema=schema_pedidos,
    partition_spec=spec_particao,
    properties={
        "write.parquet.compression-codec": "zstd",
        "write.metadata.compression-codec": "gzip",
    },
)
print(tabela.schema())
print(tabela.spec())
print("Localização:", tabela.location())

table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
}
[
  1000: data_pedido_mes: month(10)
]
Localização: s3://silver/bronze_processado/pedidos_ecommerce


In [4]:
# Célula 4 — Leitura da Bronze e primeira carga
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO["s3.endpoint"],          # http://host.docker.internal:9000 (definido na Célula 2)
    aws_access_key_id=MINIO["s3.access-key-id"],
    aws_secret_access_key=MINIO["s3.secret-access-key"],
    region_name="us-east-1",
)

arquivos = [
    o["Key"] for o in s3.list_objects_v2(Bucket="bronze", Prefix="ecommerce_sintetico/").get("Contents", [])
    if o["Key"].endswith(".parquet")
]
df_bronze = pd.read_parquet(io.BytesIO(s3.get_object(Bucket="bronze", Key=arquivos[0])["Body"].read()))
print(f"Bronze: {len(df_bronze):,} linhas em {arquivos[0]}")

COLUNAS = [campo.name for campo in tabela.schema().fields]

def preparar_lote(df: pd.DataFrame) -> pa.Table:
    """Converte um DataFrame da Bronze para o schema Arrow da tabela Iceberg."""
    df = df.copy()
    df["avaliacao_cliente"] = pd.to_numeric(df["avaliacao_cliente"], errors="coerce").astype("Int32")
    df["quantidade"] = df["quantidade"].astype("int32")
    return pa.Table.from_pandas(df[COLUNAS], schema=tabela.schema().as_arrow(), preserve_index=False)

inicio = time.time()
tabela.append(preparar_lote(df_bronze.iloc[:50_000]))
print(f"✅ Snapshot 1: 50.000 registros em {time.time() - inicio:.2f}s")

Bronze: 1,000,000 linhas em ecommerce_sintetico/ano=2026/mes=10/dia=08/pedidos.parquet
✅ Snapshot 1: 50.000 registros em 2.12s


In [5]:
# Célula 5 — Segunda carga (Snapshot 2) e histórico de snapshots
tabela.append(preparar_lote(df_bronze.iloc[50_000:100_000]))
tabela = catalog.load_table(NOME_TABELA)

def listar_snapshots(t):
    for s in sorted(t.snapshots(), key=lambda s: s.timestamp_ms):
        ts = datetime.fromtimestamp(s.timestamp_ms / 1000, tz=timezone.utc)
        print(f"{s.snapshot_id}  {ts:%H:%M:%S}  {s.summary.operation.value:<9}"
              f" +{s.summary.get('added-records', '0'):>6}  total={s.summary.get('total-records')}")

listar_snapshots(tabela)

# Alternativa tabular (metadados como tabela Arrow)
tabela.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "operation"]]

1033362292451318245  22:41:17  append    + 50000  total=50000
4931607074258359137  22:42:56  append    + 50000  total=100000


,committed_at,snapshot_id,operation
0,2026-11-05 22:41:17.501,1033362292451318245,append
1,2026-11-05 22:42:56.133,4931607074258359137,append


In [6]:
# Célula 6 — Time travel por ID e por instante
snapshots = sorted(tabela.snapshots(), key=lambda s: s.timestamp_ms)
v1 = snapshots[0]                                     # o mais antigo é o PRIMEIRO em ordem cronológica

n_v1    = tabela.scan(snapshot_id=v1.snapshot_id).to_arrow().num_rows
n_atual = tabela.scan().to_arrow().num_rows
print(f"Snapshot inicial: {n_v1:,} | Atual: {n_atual:,}")

# "Como estava a tabela neste instante?"
s = tabela.snapshot_as_of_timestamp(v1.timestamp_ms)
print("Snapshot vigente naquele instante:", s.snapshot_id)

# Leitura com filtro e projeção: o Iceberg poda partições e arquivos pelas estatísticas dos manifests
df = tabela.scan(
    row_filter="status_pedido == 'cancelado' and data_pedido >= '2022-01-15T00:00:00'",
    selected_fields=("id_pedido", "valor_total", "data_pedido"),
).to_pandas()
print(f"Cancelados após 15/01: {len(df):,}")

Snapshot inicial: 50,000 | Atual: 100,000
Snapshot vigente naquele instante: 1033362292451318245
Cancelados após 15/01: 5,996


In [7]:
# Célula 7 — Direito de eliminação (LGPD art. 18): apagar todos os pedidos de um cliente
cliente = int(tabela.scan(selected_fields=("id_cliente",), limit=1).to_pandas().iloc[0, 0])
print("Antes :", tabela.scan(row_filter=f"id_cliente == {cliente}").to_arrow().num_rows)

tabela.delete(delete_filter=f"id_cliente == {cliente}")
tabela = catalog.load_table(NOME_TABELA)

print("Depois:", tabela.scan(row_filter=f"id_cliente == {cliente}").to_arrow().num_rows)
print("Operação:", tabela.current_snapshot().summary.operation)

Antes : 2
Depois: 0
Operação: Operation.OVERWRITE


In [8]:
# Célula 8 — UPSERT (MERGE) pela chave id_pedido
lote = tabela.scan(row_filter="id_pedido <= 3").to_pandas()
lote["status_pedido"] = "reembolsado"                 # 3 pedidos alterados

novo = lote.iloc[[0]].copy()
novo["id_pedido"], novo["status_pedido"] = 2_000_001, "pendente"   # 1 pedido novo

resultado = tabela.upsert(
    pa.Table.from_pandas(pd.concat([lote, novo]), schema=tabela.schema().as_arrow(), preserve_index=False),
    join_cols=["id_pedido"],
)
print(f"Atualizados: {resultado.rows_updated} | Inseridos: {resultado.rows_inserted}")
tabela = catalog.load_table(NOME_TABELA)
listar_snapshots(tabela)

Atualizados: 3 | Inseridos: 1
1033362292451318245  22:41:17  append    + 50000  total=50000
4931607074258359137  22:42:56  append    + 50000  total=100000
4380684457971096852  22:48:15  overwrite + 89278  total=99998
957728382396595130  22:58:32  overwrite + 49996  total=99995
4173374320213541945  22:58:33  append    +     3  total=99998
1366018422625401797  22:58:33  append    +     1  total=99999


In [9]:
# Célula 9 — Adicionar e renomear colunas: operações apenas de METADADOS
with tabela.update_schema() as atualizacao:
    atualizacao.add_column("pedido_internacional", BooleanType(),
                           doc="Indica se o pedido foi feito por cliente de outro país")
    atualizacao.rename_column("avaliacao_cliente", "nota_cliente")

tabela = catalog.load_table(NOME_TABELA)
print(tabela.schema())

amostra = tabela.scan(selected_fields=("id_pedido", "nota_cliente", "pedido_internacional"), limit=3)
print(amostra.to_pandas())

table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: nota_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
  14: pedido_internacional: optional boolean (Indica se o pedido foi feito por cliente de outro país)
}
   id_pedido  nota_cliente pedido_internacional
0    2000001             5                 None
1          1             5                 None
2          2             2                 None


In [10]:
# Célula 10 — Outro motor lendo a mesma tabela (reaproveita a conexão `con` da Célula 1)
con.execute("LOAD iceberg;")             # extensão já instalada na imagem

print("metadata.json atual:", tabela.metadata_location)
con.sql(f"""
    SELECT categoria, regiao, COUNT(*) AS pedidos,
           ROUND(SUM(valor_total), 2) AS receita, ROUND(AVG(nota_cliente), 2) AS nota_media
    FROM iceberg_scan('{tabela.metadata_location}')
    WHERE status_pedido = 'concluido'
    GROUP BY ALL ORDER BY receita DESC LIMIT 10
""").df()



metadata.json atual: s3://silver/bronze_processado/pedidos_ecommerce/metadata/00005-9bab0ee2-6915-4678-902f-291092e7a4ad.metadata.json


,categoria,regiao,pedidos,receita,nota_media
0,Alimentos,Nordeste,3031,17085204.43,3.88
1,Roupas,Sul,3002,17000542.94,3.91
2,Esportes,Sul,2998,16894567.31,3.92
3,Esportes,Centro-Oeste,3077,16887176.25,3.90
4,Livros,Norte,3027,16869678.73,3.91
5,Eletrônicos,Sul,3075,16865221.11,3.93
6,Eletrônicos,Norte,3049,16810841.08,3.89
7,Roupas,Centro-Oeste,3038,16736219.59,3.93
8,Livros,Centro-Oeste,2986,16636780.68,3.89
9,Roupas,Sudeste,3006,16601599.74,3.91


In [11]:
# Célula 11 — Estrutura física: metadata.json → manifest list → manifests → dados
objetos = s3.list_objects_v2(Bucket="silver", Prefix="bronze_processado/pedidos_ecommerce/")["Contents"]

def classificar(chave: str) -> str:
    nome = chave.rsplit("/", 1)[-1]
    if nome.endswith(".metadata.json"): return "metadata.json"
    if nome.startswith("snap-"):        return "manifest list"
    if nome.endswith(".avro"):          return "manifest"
    return "dados (parquet)"

resumo = pd.DataFrame([{"tipo": classificar(o["Key"]), "kb": o["Size"] / 1024} for o in objetos])
print(resumo.groupby("tipo").agg(arquivos=("kb", "size"), total_kb=("kb", "sum")).round(1))

# Visões de metadados prontas
print(tabela.inspect.partitions().to_pandas()[["partition", "record_count", "file_count"]])
print(tabela.inspect.files().to_pandas()[["file_path", "record_count"]].head())

                 arquivos  total_kb
tipo                               
dados (parquet)         8    7320.9
manifest               10      54.5
manifest list           6      11.2
metadata.json           7      26.1
                  partition  record_count  file_count
0  {'data_pedido_mes': 624}         89279           4
1  {'data_pedido_mes': 625}         10720           1
                                           file_path  record_count
0  s3://silver/bronze_processado/pedidos_ecommerc...             1
1  s3://silver/bronze_processado/pedidos_ecommerc...             3
2  s3://silver/bronze_processado/pedidos_ecommerc...         49996
3  s3://silver/bronze_processado/pedidos_ecommerc...         39279
4  s3://silver/bronze_processado/pedidos_ecommerc...         10720


In [12]:
# Célula 12 — Agregação direta sobre a Bronze (1 milhão de pedidos)
inicio = time.time()
resultado = con.sql("""
    SELECT categoria,
           COUNT(*)                    AS total_pedidos,
           ROUND(SUM(valor_total), 2)  AS receita_total,
           ROUND(AVG(valor_total), 2)  AS ticket_medio
    FROM read_parquet('s3://bronze/ecommerce_sintetico/ano=*/**/*.parquet')
    WHERE status_pedido = 'concluido'
    GROUP BY categoria
    ORDER BY receita_total DESC
""").df()
print(f"Tempo: {time.time() - inicio:.2f}s")
resultado

Tempo: 0.72s


,categoria,total_pedidos,receita_total,ticket_medio
0,Roupas,150325,8.288938e+08,5514.01
1,Alimentos,150049,8.283236e+08,5520.35
2,Esportes,150123,8.276828e+08,5513.36
3,Eletrônicos,149950,8.272560e+08,5516.88
4,Livros,150125,8.263304e+08,5504.28


In [13]:
# Célula 13 — Cancelamento e satisfação por região (CTE + FILTER)
con.sql("""
    WITH pedidos AS (
        SELECT regiao, status_pedido, valor_total, avaliacao_cliente
        FROM read_parquet('s3://bronze/ecommerce_sintetico/ano=*/**/*.parquet')
    )
    SELECT regiao,
           COUNT(*)                                                            AS total_pedidos,
           ROUND(SUM(valor_total), 2)                                          AS receita_total,
           ROUND(100.0 * COUNT(*) FILTER (WHERE status_pedido = 'cancelado') / COUNT(*), 2)
                                                                               AS taxa_cancelamento_pct,
           ROUND(AVG(avaliacao_cliente), 2)                                    AS nota_media
    FROM pedidos
    GROUP BY regiao
    ORDER BY receita_total DESC
""").df()

,regiao,total_pedidos,receita_total,taxa_cancelamento_pct,nota_media
0,Centro-Oeste,200648,1.107246e+09,9.96,3.91
1,Sul,200055,1.105015e+09,9.87,3.90
2,Sudeste,199800,1.103193e+09,9.97,3.91
3,Norte,199943,1.099606e+09,9.98,3.91
4,Nordeste,199554,1.098873e+09,9.93,3.90


In [14]:
# Célula 14 — Crescimento mensal por categoria (window function LAG)
con.sql("""
    WITH vendas_mensais AS (
        SELECT DATE_TRUNC('month', data_pedido) AS mes, categoria,
               ROUND(SUM(valor_total), 2)       AS receita_mes
        FROM read_parquet('s3://bronze/ecommerce_sintetico/ano=*/**/*.parquet')
        WHERE status_pedido = 'concluido'
        GROUP BY ALL
    )
    SELECT mes, categoria, receita_mes,
           ROUND(100.0 * (receita_mes - LAG(receita_mes) OVER w)
                 / NULLIF(LAG(receita_mes) OVER w, 0), 2) AS crescimento_pct
    FROM vendas_mensais
    WINDOW w AS (PARTITION BY categoria ORDER BY mes)
    QUALIFY mes < '2022-04-01'
    ORDER BY categoria, mes
""").df()

,mes,categoria,receita_mes,crescimento_pct
0,2022-01-01,Alimentos,73087878.22,NaN
1,2022-02-01,Alimentos,66861956.30,-8.52
2,2022-03-01,Alimentos,74943624.38,12.09
3,2022-01-01,Eletrônicos,73343279.53,NaN
4,2022-02-01,Eletrônicos,67857475.37,-7.48
5,2022-03-01,Eletrônicos,74254134.56,9.43
6,2022-01-01,Esportes,73153747.01,NaN
7,2022-02-01,Esportes,65934309.94,-9.87
8,2022-03-01,Esportes,73474186.63,11.44
9,2022-01-01,Livros,73039897.02,NaN


In [15]:
# Célula 15 — Views no arquivo curso_analytics.duckdb + exportação para s3://gold/exploratorio
con.execute("""
    CREATE OR REPLACE VIEW vw_pedidos_bronze AS
    SELECT * FROM read_parquet('s3://bronze/ecommerce_sintetico/ano=*/**/*.parquet')
""")
con.execute("""
    CREATE OR REPLACE VIEW vw_pokemon_bronze AS
    SELECT * FROM read_parquet('s3://bronze/pokeapi/pokemon/ano=*/**/*.parquet', union_by_name = true)
""")

# Resultados exploratórios NÃO vão para a Bronze (que guarda só dados brutos)
con.execute("""
    COPY (
        SELECT categoria, regiao, DATE_TRUNC('month', data_pedido) AS mes,
               COUNT(*) AS pedidos, ROUND(SUM(valor_total), 2) AS receita
        FROM vw_pedidos_bronze
        WHERE status_pedido = 'concluido'
        GROUP BY ALL
    ) TO 's3://gold/exploratorio/resumo_mensal.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.sql("SELECT COUNT(*) AS linhas FROM 's3://gold/exploratorio/resumo_mensal.parquet'").show()
con.close()          # libera o arquivo .duckdb antes de seguir para o dbt

┌────────┐
│ linhas │
│ int64  │
├────────┤
│    300 │
└────────┘

